In [1]:
import os
from pathlib import Path
import joblib
import plotly.express as px
import plotly.graph_objects as go
import numpy as np
import pandas as pd
import torch
from dotenv import load_dotenv

load_dotenv()

from src.data_fetch import get_data
from src.preprocessing import add_features, split, preprocessing
from src.transformer import StockTransformer
from src.train import train_model
from src.evaluate import evaluate_model

LOOKBACK, HORIZON = 60, 7
EPOCHS, BATCH_SIZE, LEARNING_RATE = 80, 32, 1e-3
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
TICKER = os.getenv("STOCK")
PERIOD = f"{os.getenv("PERIOD")}"
INTERVAL = f"{os.getenv("INTERVAL")}"

print("Device:", DEVICE)

Device: cpu


In [2]:
raw_df = get_data(TICKER, PERIOD, INTERVAL)
print(raw_df.shape)
display(raw_df.head())

(522, 5)


c:\Users\kaush\Files\Codes\Projects\Mini Project\Stock-Forecasting\venv\Lib\site-packages\yfinance\utils.py:568: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  quotes.sort_index(inplace=True)
c:\Users\kaush\Files\Codes\Projects\Mini Project\Stock-Forecasting\venv\Lib\site-packages\yfinance\utils.py:604: Pandas4Warning: The inplace keyword in DataFrame.set_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  splits.set_index("date", inplace=True)
c:\Users\kaush\Files\Codes\Projects\Mini Project\Stock-Forecasting\venv\Lib\site-packages\yfinance\utils.py:606: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.

,Open,High,Low,Close,Volume
Date,,,,,
2016-10-31,13.499333,13.499333,12.397333,12.704000,357070500
2016-11-07,12.906000,13.166000,12.028000,12.570667,390744000
2016-11-14,12.533333,12.866667,11.879333,12.334667,359790000
2016-11-21,12.336000,13.149333,12.247333,13.110000,258336000
2016-11-28,13.032000,13.290000,12.000000,12.098000,325264500


In [3]:
fig = px.line(
    x=raw_df.index,
    y=raw_df.Close,
    title=f'{TICKER} adjusted closing price',
)

fig.update_layout(
    xaxis_title="Date",
    yaxis_title="Price"
)

fig.show()

## Preprocessing

### Feature Engeinnering

In [4]:
data = add_features(raw_df)
display(data.head())
print("Shape:", data.shape)

,Open,High,Low,Close,Volume,Return,MA_5,MA_20,Volatility_10
Date,,,,,,,,,
2017-03-13,16.321333,17.716667,16.185333,17.433332,443722500,0.073085,17.146533,15.116500,0.042913
2017-03-20,17.373333,17.653334,16.682667,17.544001,353245500,0.006348,17.025600,15.358500,0.042108
2017-03-27,17.373333,18.799999,17.316668,18.553333,380064000,0.057531,17.309600,15.657633,0.044356
2017-04-03,19.126667,20.325333,18.972000,20.169333,630064500,0.087100,17.989200,16.049367,0.049890
2017-04-10,20.610001,20.915333,19.686666,20.266666,430866000,0.004826,18.793333,16.407200,0.049374


Shape: (503, 9)


### Splitting the dataset

In [5]:
train_df, val_df, test_df = split(data, train_size=0.70, val_size=0.10)
print("Train:", train_df.shape, train_df.index.min(), "to", train_df.index.max())
print("Validation:", val_df.shape, val_df.index.min(), "to", val_df.index.max())
print("Test:", test_df.shape, test_df.index.min(), "to", test_df.index.max())

Train: (352, 9) 2017-03-13 00:00:00 to 2023-12-04 00:00:00
Validation: (50, 9) 2023-12-11 00:00:00 to 2024-11-18 00:00:00
Test: (101, 9) 2024-11-25 00:00:00 to 2026-10-26 00:00:00


In [6]:
((X_train, y_train), (X_val, y_val), (X_test, y_test), x_scaler, y_scaler, feature_names) = preprocessing(
    train_df, val_df, test_df, lookback=LOOKBACK, horizon=HORIZON
)
print("Features:", feature_names)
print("Train:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)

Features: ['Open', 'High', 'Low', 'Close', 'Volume', 'Return', 'MA_5', 'MA_20', 'Volatility_10']
Train: (286, 60, 9) (286, 7)
Validation: (44, 60, 9) (44, 7)
Test: (95, 60, 9) (95, 7)


## Model Training

In [7]:
model = StockTransformer(
    n_features=X_train.shape[-1], horizon=HORIZON,
    d_model=64, nhead=4, num_layers=3,
    dim_feedforward=128, dropout=0.1
)
print(model)

StockTransformer(
  (input_projection): Linear(in_features=9, out_features=64, bias=True)
  (position): PositionalEncoding()
  (encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-2): 3 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=64, out_features=64, bias=True)
        )
        (linear1): Linear(in_features=64, out_features=128, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=128, out_features=64, bias=True)
        (norm1): LayerNorm((64,), eps=1e-05, elementwise_affine=True, bias=True)
        (norm2): LayerNorm((64,), eps=1e-05, elementwise_affine=True, bias=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (norm): LayerNorm((64,), eps=1e-05, elementwise_affine=True, bias=True)
  (head): Sequential(
    (0): Linear(in_features=64, out_features=32, bias=True

c:\Users\kaush\Files\Codes\Projects\Mini Project\Stock-Forecasting\ml\src\transformer.py:40: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(layers, num_layers=num_layers)


In [8]:
model, history = train_model(
    model, X_train, y_train, X_val, y_val,
    epochs=EPOCHS, batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE, patience=12, device=DEVICE
)

Epoch 001/80 | train=0.083809 | val=0.032958
Epoch 002/80 | train=0.022427 | val=0.015580
Epoch 003/80 | train=0.017757 | val=0.009450
Epoch 004/80 | train=0.014636 | val=0.008112
Epoch 005/80 | train=0.014586 | val=0.008515
Epoch 006/80 | train=0.012861 | val=0.008507
Epoch 007/80 | train=0.012239 | val=0.006630
Epoch 008/80 | train=0.012920 | val=0.009511
Epoch 009/80 | train=0.010756 | val=0.022168
Epoch 010/80 | train=0.011377 | val=0.006980
Epoch 011/80 | train=0.011732 | val=0.008983
Epoch 012/80 | train=0.009441 | val=0.008648
Epoch 013/80 | train=0.010397 | val=0.010179
Epoch 014/80 | train=0.009118 | val=0.008393
Epoch 015/80 | train=0.009460 | val=0.008028
Epoch 016/80 | train=0.009258 | val=0.011220
Epoch 017/80 | train=0.009346 | val=0.010227
Epoch 018/80 | train=0.008547 | val=0.008977
Epoch 019/80 | train=0.008191 | val=0.010964
Early stopping.


## Model Evaluation

In [9]:
fig = go.Figure()
fig.add_trace(go.Scatter(y=history["train_loss"], mode="lines", name="Train"))
fig.add_trace(go.Scatter(y=history["val_loss"], mode="lines", name="Validation"))

fig.update_layout(
    width=900, height=400,
    title="Training history",
    xaxis_title="Epoch",
    yaxis_title="MSE loss",
    template="plotly_white",
)
fig.update_xaxes(showgrid=True, gridcolor="rgba(0,0,0,0.25)")
fig.update_yaxes(showgrid=True, gridcolor="rgba(0,0,0,0.25)")

fig.show()

In [10]:
metrics, actual, predicted = evaluate_model(
    model, X_test, y_test, y_scaler, device=DEVICE
)
display(pd.Series(metrics, name="Test metrics"))

MAE             88.891251
RMSE            98.685867
MAPE_percent    22.788836
Name: Test metrics, dtype: float64

In [11]:
n = min(250, len(actual))

fig = go.Figure()
fig.add_trace(go.Scatter(y=actual[:n], mode="lines", name="Actual"))
fig.add_trace(go.Scatter(y=predicted[:n], mode="lines", name="Predicted"))

fig.update_layout(
    width=1200, height=400,
    title="Test predictions (forecast points pooled across horizons)",
    xaxis_title="Forecast point",
    yaxis_title="Adjusted close",
    template="plotly_white",
)
fig.update_xaxes(showgrid=True, gridcolor="rgba(0,0,0,0.25)")
fig.update_yaxes(showgrid=True, gridcolor="rgba(0,0,0,0.25)")

fig.show()

In [ ]:
def forecast_next_sessions(model, feature_df, x_scaler, y_scaler, feature_names, lookback=60, horizon=7, device="cpu"):
    latest = x_scaler.transform(feature_df[feature_names]).astype(np.float32)[-lookback:]
    model.eval()
    with torch.no_grad():
        scaled = model(torch.tensor(latest[None, :, :], dtype=torch.float32).to(device)).cpu().numpy()
    prices = y_scaler.inverse_transform(scaled.reshape(-1, 1)).ravel()
    dates = pd.bdate_range(feature_df.index[-1] + pd.Timedelta(days=1), periods=horizon)
    return pd.DataFrame({"Forecast_Date": dates, "Predicted_Close": prices})

forecast_df = forecast_next_sessions(
    model, data, x_scaler, y_scaler, feature_names, LOOKBACK, HORIZON, DEVICE
)
display(forecast_df)

,Forecast_Date,Predicted_Close
0,2026-10-27,312.863190
1,2026-10-28,320.020813
2,2026-10-29,290.557312
3,2026-10-30,298.893219
4,2026-11-02,307.117889
5,2026-11-03,274.061218
6,2026-11-04,297.693054


## Saving the model for later use

In [13]:
out = Path("outputs")
out.mkdir(exist_ok=True)
ticker = TICKER.replace(".", "_").replace("^", "")

torch.save({
    "model_state_dict": model.cpu().state_dict(),
    "n_features": X_train.shape[-1],
    "horizon": HORIZON,
    "lookback": LOOKBACK,
    "feature_names": feature_names,
    "ticker": TICKER
}, out / f"{ticker}_transformer.pth")

joblib.dump({"x_scaler": x_scaler, "y_scaler": y_scaler}, out / f"{ticker}_scalers.joblib")
raw_df.to_csv(out / f"{ticker}_historical.csv")
forecast_df.to_csv(out / f"{ticker}_forecast.csv", index=False)
print("Saved to:", out.resolve())

Saved to: C:\Users\kaush\Files\Codes\Projects\Mini Project\Stock-Forecasting\ml\outputs
